In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/04_Evaluation/01_data_generate.ipynb/data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
df_ground_truth.head()

,question,document
0,Can I join the course late if I just found out...,74eb249bbf
1,"If I start the course now, is it still okay to...",74eb249bbf
2,Do I have to finish everything before a certai...,74eb249bbf
3,Can I still get a certificate if I join after ...,74eb249bbf
4,What’s the deadline for submitting the project...,74eb249bbf


In [3]:

ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
ground_truth[10]

{'question': 'Where can I watch the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.'},
 {'id': 'db78580409

In [7]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [8]:
q = ground_truth[0]
q

{'question': 'Can I join the course late if I just found out about it?',
 'document': '74eb249bbf'}

In [9]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [10]:
results = text_search(q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp

In [11]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
977bf7786c == 74eb249bbf: False
04919992b3 == 74eb249bbf: False
69d122f12e == 74eb249bbf: False
193612db63 == 74eb249bbf: False


In [12]:
text_search(q["question"])

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp

In [13]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [14]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [15]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Can I join the course late if I just found out about it?


[1, 0, 0, 0, 0]

In [16]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Do students get the Zoom link for the live session, or is it only for the teaching staff?


[1, 0, 0, 0, 0]

In [17]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

How do I keep track of the LLM Zoomcamp syllabus and deadlines in one place?


[1, 0, 0, 0, 0]

In [18]:
[0, 0, 0, 0, 0]

[0, 0, 0, 0, 0]

In [19]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [20]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [21]:
relevance[:15]

[1, 0, 0, 0, 0]

In [22]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [23]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [24]:
sample = relevance_total = compute_relevance_total(ground_truth_sample, text_search)


  0%|          | 0/15 [00:00<?, ?it/s]

In [25]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

In [26]:

sample = relevance_total[:15]

In [27]:
sample

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [28]:
14 / 15

0.9333333333333333

## Search Evaluation Metrics

In [29]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.8666666666666667

In [30]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [31]:
hit_rate

<function __main__.hit_rate(relevance)>

In [32]:
hit_rate(relevance_total)

0.8593220338983051

In [33]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.6688888888888889

In [34]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [35]:
mrr(sample)

0.6688888888888889

In [36]:
	
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [37]:
evaluate(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8593220338983051, 'mrr': 0.7340395480225986}

In [38]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [39]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8796610169491526, 'mrr': 0.755423728813559}

In [40]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [41]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/590 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8949152542372881, 'mrr': 0.7987570621468925}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8949152542372881, 'mrr': 0.789152542372881}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8593220338983051, 'mrr': 0.7340395480225986}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8322033898305085, 'mrr': 0.7051694915254235}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8084745762711865, 'mrr': 0.6826553672316381}


In [42]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [43]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/590 [00:00<?, ?it/s]

In [44]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
35,5.0,10.0,0.5,0.976271,0.877373
19,2.0,4.0,0.2,0.976271,0.877373
3,1.0,2.0,0.1,0.976271,0.877373
4,1.0,2.0,0.2,0.974576,0.875480
18,2.0,4.0,0.1,0.974576,0.875113
34,5.0,10.0,0.2,0.974576,0.874831
33,5.0,10.0,0.1,0.974576,0.873051
7,1.0,4.0,0.2,0.966102,0.869718
20,2.0,4.0,0.5,0.972881,0.868644
6,1.0,4.0,0.1,0.964407,0.868249
